# Preprocessing Pipeline - BLOCK-5

**Objective**: Implement a deterministic, time-aware preprocessing pipeline for the delivery-time prediction system.

**Inputs**: `Dataset/GOLD_STANDARD_DATASET.csv` (and `Dataset/final_dataset_CLEANED.csv` for date recovery)
**Outputs**: `preprocessing_pipeline_block5.pkl`, Processed Matrices

## 1. Introduction & Objective
We will load the dataset, strictly sort by time to prevent leakage, define explicit feature groups, split the data 70/15/15 temporally, and fit an sklearn pipeline.

## 2. Data Loading & Sorting
**Note**: `GOLD_STANDARD_DATASET.csv` is missing `order_date`. We merge it from `final_dataset_CLEANED.csv` based on row alignment.

In [10]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
import joblib
import os

# File Paths
dataset_path = 'Dataset/GOLD_STANDARD_DATASET.csv'
date_source_path = 'Dataset/final_dataset_CLEANED.csv'

# Load Main Dataset

print("Loading datasets...")
df = pd.read_csv(dataset_path)

# Load Date Column (Data Fix)
df_date = pd.read_csv(date_source_path, usecols=['order_date'])

# Merge Date
df['order_date'] = df_date['order_date']

# Convert to Datetime
df['order_date'] = pd.to_datetime(df['order_date'])

# Sort Strictly by Time
df = df.sort_values(by='order_date').reset_index(drop=True)

print(f"Data Loaded and Sorted. Shape: {df.shape}")
print(f"Date Range: {df['order_date'].min()} to {df['order_date'].max()}")

Loading datasets...
Data Loaded and Sorted. Shape: (69512, 27)
Date Range: 2025-11-06 00:00:00 to 2026-01-05 00:00:00


## 3. Feature Group Definitions
Explicitly defining Numeric, Binary, and One-Hot features.

In [11]:
# Target & Time
TARGET = "delivery_time_hours"
TIME_COLUMN = "order_date"

# Feature Groups
NUMERIC_FEATURES = [
    'expected_time_no_traffic', 
    'vehicle_distance_mismatch', 
    'vehicle_traffic_stress', 
    'distance_km', 
    'vehicle_time_efficiency', 
    'traffic_weather_risk', 
    'operational_stress_index', 
    'route_frequency', 
    'route_traffic_volatility', 
    'temperature',
    'destination_city_encoded', 
    'route_id_encoded'
]

BINARY_FEATURES = [
    'is_peak_hour', 
    'is_heavy_traffic_truck', 
    'is_long_route'
]

ONE_HOT_FEATURES = [
    'traffic_level_medium', 
    'vehicle_type_car', 
    'vehicle_type_motorcycle', 
    'vehicle_type_pickup', 
    'vehicle_type_truck', 
    'vehicle_type_van', 
    'weather_clear', 
    'weather_clouds', 
    'weather_haze', 
    'weather_mist'
]

ALL_FEATURES = NUMERIC_FEATURES + BINARY_FEATURES + ONE_HOT_FEATURES

# Consistency Check
missing_cols = [c for c in ALL_FEATURES + [TARGET] if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

print(f"Total Features: {len(ALL_FEATURES)}")

Total Features: 25


## 4. Time-Aware Splitting
Splitting 70/15/15 strictly by time index.

In [13]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

# Verify Temporal Integrity
print(f"Train Dates: {train_df[TIME_COLUMN].min()} -> {train_df[TIME_COLUMN].max()}")
print(f"Val Dates:   {val_df[TIME_COLUMN].min()} -> {val_df[TIME_COLUMN].max()}")
print(f"Test Dates:  {test_df[TIME_COLUMN].min()} -> {test_df[TIME_COLUMN].max()}")

assert train_df[TIME_COLUMN].max() <= val_df[TIME_COLUMN].min(), "Train/Val Temporal Leakage!"
assert val_df[TIME_COLUMN].max() <= test_df[TIME_COLUMN].min(), "Val/Test Temporal Leakage!"

X_train = train_df[ALL_FEATURES]
y_train = train_df[TARGET]
X_val = val_df[ALL_FEATURES]
y_val = val_df[TARGET]
X_test = test_df[ALL_FEATURES]
y_test = test_df[TARGET]

Train Dates: 2025-11-06 00:00:00 -> 2025-12-18 00:00:00
Val Dates:   2025-12-18 00:00:00 -> 2025-12-27 00:00:00
Test Dates:  2025-12-27 00:00:00 -> 2026-01-05 00:00:00


## 5. Pipeline Construction
Using `ColumnTransformer` to apply `StandardScaler` to numeric features and `passthrough` to others.

In [14]:
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUMERIC_FEATURES),
        ('passthrough', 'passthrough', BINARY_FEATURES + ONE_HOT_FEATURES)
    ],
    remainder='drop'
)

pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

## 6. Pipeline Fitting & Transformation

In [15]:
print("Fitting pipeline on Training Data...")
pipeline.fit(X_train, y_train)

print("Transforming splits...")
X_train_processed = pipeline.transform(X_train)
X_val_processed = pipeline.transform(X_val)
X_test_processed = pipeline.transform(X_test)

print(f"Processed Train Shape: {X_train_processed.shape}")
print(f"Processed Val Shape: {X_val_processed.shape}")
print(f"Processed Test Shape: {X_test_processed.shape}")

Fitting pipeline on Training Data...
Transforming splits...
Processed Train Shape: (48658, 25)
Processed Val Shape: (10427, 25)
Processed Test Shape: (10427, 25)


## 7. Artifact Saving

In [16]:
# Save Pipeline
joblib.dump(pipeline, 'preprocessing_pipeline_block5.pkl')
print("Pipeline saved to 'preprocessing_pipeline_block5.pkl'")

# Optional: Save processed matrices (as numpy arrays or dataframes)
# np.save('X_train_processed.npy', X_train_processed)
# np.save('X_val_processed.npy', X_val_processed)
# np.save('X_test_processed.npy', X_test_processed)
# np.save('y_train.npy', y_train.to_numpy())
# np.save('y_val.npy', y_val.to_numpy())
# np.save('y_test.npy', y_test.to_numpy())

Pipeline saved to 'preprocessing_pipeline_block5.pkl'


## 8. Validation Summary

In [18]:
print("Validation Checks:")
print("1. Shapes Check:", 
      X_train_processed.shape[0] == len(y_train), 
      X_val_processed.shape[0] == len(y_val), 
      X_test_processed.shape[0] == len(y_test))

print("2. NaN Check:", 
      np.isnan(X_train_processed).sum() == 0, 
      np.isnan(X_val_processed).sum() == 0, 
      np.isnan(X_test_processed).sum() == 0)

expected_feats = len(NUMERIC_FEATURES) + len(BINARY_FEATURES) + len(ONE_HOT_FEATURES)
print("3. Feature Count Check:", X_train_processed.shape[1] == expected_feats, 
      f"Expected: {expected_feats}, Got: {X_train_processed.shape[1]}")

print("\nPipeline Implementation Complete.")

Validation Checks:
1. Shapes Check: True True True
2. NaN Check: True True True
3. Feature Count Check: True Expected: 25, Got: 25

Pipeline Implementation Complete.
